# 05 — Model Training

ML/DL model training, evaluation, and comparison across LSTM, Transformer, and XGBoost.

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

## 1. Load Feature Data

In [ ]:
from src.data.price_fetcher import PriceFetcher
from src.features.pipeline import FeaturePipeline

fetcher = PriceFetcher()
wti_df = fetcher.fetch_single('wti', start='2018-01-01')

pipeline = FeaturePipeline()
feature_df = pipeline.build(price_df=wti_df)
target = pipeline.create_targets(feature_df, horizon=5)
feature_df['target'] = target
feature_df = feature_df.dropna(subset=['target'])

feature_cols = [c for c in feature_df.columns 
                if c not in {'target', 'Open', 'High', 'Low', 'Close', 'Volume'}]
X = feature_df[feature_cols].fillna(0).values
y = feature_df['target'].values
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features")

## 2. Train XGBoost Model

In [ ]:
from src.models.xgboost_model import XGBoostModel
from src.models.training import ModelTrainer

xgb = XGBoostModel()
trainer = ModelTrainer(xgb, n_folds=3)
result = trainer.walk_forward_validate(X, y)
print(result.summary())

## 3. Feature Importance

In [ ]:
trainer.final_train(X, y)
importance = xgb.get_feature_importance(feature_names=feature_cols)
print("Top 10 most important features:")
print(importance.head(10))